# FREUID baseline — full-data training (IDE -> Colab GPU remote kernel)

Same setup as `run_smoke.ipynb`, but trains on the **full dataset (~69k train images)** instead of the
100-image smoke set: train -> infer -> submit. Compute runs on the Colab GPU; output shows in your IDE.

> ⚠️ **Time warning:** full data x baseline 20 epochs takes **several hours** on a T4, and free Colab
> disconnects after 90 min idle / ~12 h max — you may not finish in one session. For a first complete run,
> prefer the reduced-epoch variant (the `baseline_colab` cell in STEP 2). **Running STEP 1 [Colab 4]
> (Drive mount) auto-saves `checkpoints/` to Drive**, so the best-so-far model survives a disconnect.

---

## STEP 1 — on Colab (browser; bring up the tunnel)

New Colab notebook -> Runtime type **GPU (T4)** -> add 🔑 Secrets `NGROK_AUTH_TOKEN`, `KAGGLE_USERNAME`,
`KAGGLE_KEY`. Paste the cells below into Colab and run them in order.

```python
# [Colab 1] check GPU + clone + install
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU -> set Runtime to GPU (T4)')
!pip install -q pyngrok
!git clone --depth 1 https://github.com/hyunsoochung-portfolio/kaggle_freuid_2026.git
%cd kaggle_freuid_2026
!pip install -q -e .   # reuse Colab's built-in torch (CUDA); installs only timm/kaggle/etc.
```

```python
# [Colab 2] Kaggle auth (use Secrets KAGGLE_USERNAME / KAGGLE_KEY — no hardcoded tokens)
import os, json
from google.colab import userdata
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': userdata.get('KAGGLE_USERNAME'), 'key': userdata.get('KAGGLE_KEY')}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('kaggle.json written')
# ⚠️ Accept the competition rules ('Join Competition') on the comp page first, or downloads 403
```

```python
# [Colab 3] download + unzip the real competition data (creates data/train/train, public_test/...)
!kaggle competitions download -c the-freuid-challenge-2026-ijcai-ecai -p data
!cd data && unzip -q -o '*.zip' && rm -f *.zip
!echo 'train imgs:' && ls data/train/train | wc -l && ls data
```

```python
# [Colab 4] (required) mount Google Drive -> checkpoints/, submissions/ saves auto-land in Drive
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/freuid/checkpoints', exist_ok=True)
os.makedirs('/content/drive/MyDrive/freuid/submissions', exist_ok=True)
# symlink the repo's checkpoints/, submissions/ to the Drive folders so saves go straight to Drive
!rm -rf checkpoints submissions
!ln -s /content/drive/MyDrive/freuid/checkpoints checkpoints
!ln -s /content/drive/MyDrive/freuid/submissions submissions
print('checkpoints/, submissions/ -> linked to Drive (saves now go to MyDrive/freuid/)')
```

```python
# [Colab 5] GPU kernel + ngrok tunnel -> print URL
import subprocess, time, socket
from google.colab import userdata
from pyngrok import conf, ngrok
conf.get_default().auth_token = userdata.get('NGROK_AUTH_TOKEN').strip()
JUPYTER_TOKEN = 'freuid'
for t in ngrok.get_tunnels(): ngrok.disconnect(t.public_url)
ngrok.kill()
log = open('/tmp/jl.log', 'w')
subprocess.Popen(['jupyter','server','--ip=0.0.0.0','--port=8888','--no-browser','--allow-root',
    f'--ServerApp.token={JUPYTER_TOKEN}','--ServerApp.password=','--ServerApp.allow_origin=*',
    '--ServerApp.disable_check_xsrf=True','--ServerApp.allow_remote_access=True',
    '--ServerApp.root_dir=/content/kaggle_freuid_2026'], stdout=log, stderr=subprocess.STDOUT)
for i in range(30):
    time.sleep(1)
    s = socket.socket()
    if s.connect_ex(('127.0.0.1', 8888)) == 0: s.close(); break
    s.close()
print(ngrok.connect(8888, 'http').public_url + '/?token=' + JUPYTER_TOKEN)
```

-> Copy the printed `https://....ngrok-free.app/?token=freuid`. (Keep this Colab tab open — closing it ends the session.)

---

## STEP 2 — in my IDE (this notebook)
1. Top-right **kernel picker** -> `Select Another Kernel...` -> `Existing Jupyter Server...` -> paste the URL -> `Python 3`
2. Run the cells below (all run on the Colab GPU)

In [4]:
# Connection check: are we on the Colab GPU + is Drive linked?
import torch, os
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu (<- runtime is not GPU!)')
print('cwd:', os.getcwd())
print('checkpoints -> Drive?', os.path.realpath('checkpoints'))  # /content/drive/... means auto-save is on
!ls data && echo 'train imgs:' && ls data/train/train | wc -l

device: cuda
cwd: /content/kaggle_freuid_2026
checkpoints -> Drive? /content/drive/MyDrive/freuid/checkpoints
public_test	       train		 train_sample
sample_submission.csv  train_labels.csv  train_sample_labels.csv
train imgs:
69352


### (optional) Colab-friendly config — fewer epochs to finish in one session
Same full data, only epochs reduced. **Recommended for a first complete run.** To run the real baseline
(20 epochs), skip this cell and set the training cell's config to `configs/baseline.yaml`.

In [10]:
%%writefile configs/a100_effnetv2m.yaml
name: a100_effnetv2m
seed: 42
data_dir: data
image_size: 384
val_fraction: 0.1
backbone: tf_efficientnetv2_m.in21k
pretrained: true
epochs: 20
batch_size: 32
lr: 0.0003
weight_decay: 0.0001
num_workers: 4

Overwriting configs/a100_effnetv2m.yaml


In [11]:
# Real-data training (Colab GPU). Use baseline_colab if you wrote that cell, else baseline.
# Save path is decided by train.py from the config name: checkpoints/<name>.pt -> auto-saved to Drive if linked.
CONFIG = 'configs/a100_effnetv2m.yaml'   # or 'configs/baseline.yaml'
!python -m freuid.train --config {CONFIG}

[train] config 'a100_effnetv2m' | device=cuda | backbone=tf_efficientnetv2_m.in21k | image_size=384 mean=(0.5, 0.5, 0.5)
[train] train=62417 val=6935
epoch  1: train_loss=0.1951 val_loss=0.0034 AuDET=0.0000 APCER@1%BPCER=0.0000
  ↳ saved checkpoints/a100_effnetv2m.pt (AuDET=0.0000)
epoch  2: train_loss=0.0290 val_loss=0.0015 AuDET=0.0000 APCER@1%BPCER=0.0000
  ↳ saved checkpoints/a100_effnetv2m.pt (AuDET=0.0000)
epoch  3: train_loss=0.0229 val_loss=0.0155 AuDET=0.0002 APCER@1%BPCER=0.0010
epoch  4: train_loss=0.0210 val_loss=0.0011 AuDET=0.0000 APCER@1%BPCER=0.0000
epoch  5: train_loss=0.0282 val_loss=0.0122 AuDET=0.0002 APCER@1%BPCER=0.0014
epoch  6: train_loss=0.0167 val_loss=0.0009 AuDET=0.0000 APCER@1%BPCER=0.0000
  ↳ saved checkpoints/a100_effnetv2m.pt (AuDET=0.0000)
epoch  7: train_loss=0.0122 val_loss=0.0001 AuDET=0.0000 APCER@1%BPCER=0.0000
epoch  8: train_loss=0.0117 val_loss=0.0014 AuDET=0.0000 APCER@1%BPCER=0.0000
epoch  9: train_loss=0.0122 val_loss=0.0016 AuDET=0.0000 APCE

In [12]:
# Inference -> submission csv (checkpoint name follows the config name: baseline_colab.pt or baseline.pt)
CKPT = 'checkpoints/a100_effnetv2m.pt'   # or 'checkpoints/baseline.pt'
OUT = 'submissions/a100_effnetv2m.csv'
!python -m freuid.infer --checkpoint {CKPT} --out {OUT}
!head -3 {OUT} && wc -l {OUT}

[infer] backbone=tf_efficientnetv2_m.in21k image_size=384 (from checkpoint) | device=cuda | data_dir=data
[infer] 142818 ids total; 7821 images present locally
[infer] wrote 142818 rows -> submissions/a100_effnetv2m.csv (7821 scored, 134997 defaulted)
id,label
0009cd06cf8c426789aa58b2b05c60d2,9.138114478446369e-07
000c9f83f98c4babac3f030df5300a8e,7.869862747611478e-05
142819 submissions/a100_effnetv2m.csv


In [ ]:
# (optional) Submit to Kaggle directly from Colab (no need to move files to your machine)
OUT = 'submissions/a100_effnetv2m.csv'
!kaggle competitions submit -c the-freuid-challenge-2026-ijcai-ecai -f {OUT} -m 'colab baseline'

[Errno 2] No such file or directory: 'submissions/baseline_colab.csv'


In [2]:
CONFIG = 'configs/dylan_simple_resnet18.yaml'
!python -m freuid.train --config {CONFIG}

[train] config 'dylan_simple_resnet18' | device=cuda | backbone=resnet18 | image_size=224 mean=(0.485, 0.456, 0.406)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
[train] train=62417 val=6935
model.safetensors: 100% 46.8M/46.8M [00:01<00:00, 32.3MB/s]
  0% 0/1950 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware tha

: 

## STEP 3 — (optional) auto-receive results into my local repo too

STEP 1·2 above are the **Colab/Drive side**. Here we set up the **local side** so the results Colab saved to
Drive (`checkpoints/*.pt`) **show up automatically inside this repo folder on my Mac**.

> ⚠️ This is not a repo code change — it is **a few shell commands each person runs once on their own Mac**.
> Teammates run the same with their own Mac + Drive path.
> (The `checkpoints` ignore rule is already in the shared `.gitignore`, so no git config is needed.)

**1. Install Google Drive desktop + sign in** (in the Mac terminal)
```bash
brew install --cask google-drive      # install (Mac admin password once)
# Open the app -> sign in with the SAME Google account you use for Colab
# Settings (gear) -> prefer "Mirror files" (files actually land on local disk)
```

**2. Make the freuid folder in Drive and symlink the repo's `checkpoints` to it**
```bash
# <email> is your account. Mind the localized My Drive folder name (en: "My Drive", ko: "내 드라이브")
DRIVE="$HOME/Library/CloudStorage/GoogleDrive-<email>/My Drive/freuid"
mkdir -p "$DRIVE/checkpoints"

cd <this-repo-path>          # e.g. ~/kaggle_freuid_2026
rmdir checkpoints 2>/dev/null   # remove the existing empty checkpoints/ (only if empty)
ln -s "$DRIVE/checkpoints" checkpoints   # checkpoints -> Drive folder
```

`checkpoints` is already ignored by the shared `.gitignore`, so **this symlink does not show up in `git status`** (no extra setup).

**Verify**
```bash
realpath checkpoints   # OK if it points to /Users/<me>/Library/CloudStorage/.../freuid/checkpoints
git status --short     # nothing checkpoints-related should appear
```

-> From now on: **Colab training -> Drive -> Mac sync -> `checkpoints/baseline_colab.pt` appears in this repo.**

### How to get submissions into the repo?
`submissions/` has a git-tracked `.gitkeep` (keeps the empty folder, shared by all contributors). Turning it
into a Drive symlink like checkpoints would drop that `.gitkeep` and **change the shared repo**, so we don't.
Instead, to see the submission csv inside the repo's `submissions/`, use the copy helper (auto-detects the
Drive path; copies are gitignored, never committed):

```bash
bash scripts/pull_submissions.sh
```

(The csv also stays in the Drive folder `MyDrive/freuid/submissions/` and in the Kaggle submission history.)